# Generic league analysis

Run the function cell, then edit/run the final call for any league.

Required: precomputed division-series results CSV. Optional: division membership CSV for validation. Team-name files and raw game data are upstream inputs and are not needed here. Divisions and league labels come from the CSVs.

Scores count series wins plus `tie_score` per tied series. Default `0.5` matches the Premier League notebook; use `0` to reproduce the uploaded MLB/NBA/NFL notebooks. Teams sort by decreasing score, with alphabetical tie-breaking. Strict upsets require a lower-score series winner; weak upsets additionally count tied series between different-score teams as half an upset. Equal-score pairs never count as upsets.

`num_games` counts unordered team pairs (series), not individual fixtures. Frequencies divide by `num_admissible`; zero denominators export as blank. Incomplete round robins, duplicate pairs, nonuniform games per pair, and membership mismatches raise errors instead of silently producing misleading denominators.

Only pandas is required. Paths are relative to the notebook working directory.


Note: This is mainly copied from the existing datascripts just mashed together in one big helper file for quick use

In [2]:
import pandas as pd
from pathlib import Path
from collections import Counter
from math import comb


def generate_analysis(series_file, output_file, membership_file=None, tie_score=0.5):
    """Analyze one league's precomputed head-to-head series.

    tie_score=0.5: tied series give each team half a scoring point.
    tie_score=0: reproduce the current MLB/NBA/NFL notebooks.
    Frequencies use only pairs with different final series scores.
    """
    if tie_score not in (0, 0.5):
        raise ValueError("tie_score must be 0 or 0.5")

    keys = ["season", "league", "division"]
    series_df = pd.read_csv(series_file)
    required = keys + ["team1", "team2", "result", "games_played"]
    missing = set(required) - set(series_df.columns)
    if missing:
        raise ValueError(f"Missing series columns: {sorted(missing)}")
    if series_df.empty or series_df[required].isna().any().any():
        raise ValueError("Series data is empty or has missing required values")

    # Validate winner labels before converting to 1 / -1 / 0.
    valid = (series_df["result"].eq("TIE")
             | series_df["result"].eq(series_df["team1"])
             | series_df["result"].eq(series_df["team2"]))
    if not valid.all():
        raise ValueError("Each result must be team1, team2, or TIE")

    series_df["games_played"] = pd.to_numeric(series_df["games_played"], errors="raise")
    games = series_df["games_played"]
    if ((games <= 0) | (games % 1 != 0)).any():
        raise ValueError("games_played must contain positive integers")

    # Membership is optional, but detects omitted teams and seasons.
    memberships = None
    if membership_file is not None:
        membership_df = pd.read_csv(membership_file)
        missing = set(keys + ["teams"]) - set(membership_df.columns)
        if missing:
            raise ValueError(f"Missing membership columns: {sorted(missing)}")
        if membership_df[keys + ["teams"]].isna().any().any():
            raise ValueError("Missing membership values")
        if membership_df.duplicated(keys).any():
            raise ValueError("Duplicate membership groups")
        memberships = {}
        for _, row in membership_df.iterrows():
            teams = [t.strip() for t in row["teams"].strip().strip("()").split(",")]
            if len(teams) < 2 or not all(teams) or len(set(teams)) != len(teams):
                raise ValueError("Invalid team membership")
            memberships[tuple(row[k] for k in keys)] = set(teams)

    # One row must represent one series for each unordered team pair.
    observed_groups = set()
    for key, group in series_df.groupby(keys):
        observed_groups.add(key)
        teams = set(group["team1"]) | set(group["team2"])
        pairs = [tuple(sorted(pair)) for pair in zip(group["team1"], group["team2"])]
        if any(a == b for a, b in pairs) or len(set(pairs)) != len(pairs):
            raise ValueError(f"{key}: duplicate pairs or self-pairs")
        if len(pairs) != comb(len(teams), 2):
            raise ValueError(f"{key}: incomplete round robin")
        if memberships is not None and memberships.get(key) != teams:
            raise ValueError(f"{key}: series teams do not match membership")
        if group["games_played"].nunique() != 1:
            raise ValueError(f"{key}: games per pair are not uniform")
    if memberships is not None and set(memberships) != observed_groups:
        raise ValueError("Membership and series files contain different season/division groups")

    def get_series_result(row):
        if row["result"] == "TIE":
            return 0
        return 1 if row["result"] == row["team1"] else -1

    series_df["result"] = series_df.apply(get_series_result, axis=1)

    # Same team1/team2 scoring architecture as the existing notebooks.
    team1 = series_df[keys + ["team1", "result"]].copy()
    team1 = team1.rename(columns={"team1": "team"})
    team1["wins"] = (team1["result"].eq(1).astype(float)
                     + tie_score * team1["result"].eq(0))

    team2 = series_df[keys + ["team2", "result"]].copy()
    team2 = team2.rename(columns={"team2": "team"})
    team2["wins"] = (team2["result"].eq(-1).astype(float)
                     + tie_score * team2["result"].eq(0))

    score_sequences = pd.concat([team1, team2], ignore_index=True)
    score_sequences = score_sequences.groupby(keys + ["team"], as_index=False)["wins"].sum()
    score_lookup = score_sequences.set_index(keys + ["team"])["wins"].to_dict()

    def get_upsets(row):
        key = tuple(row[k] for k in keys)
        s1 = score_lookup[key + (row["team1"],)]
        s2 = score_lookup[key + (row["team2"],)]
        if s1 == s2:
            return 0, 0.0
        if row["result"] == 0:
            return 0, 0.5
        strict = int((row["result"] == 1 and s1 < s2)
                     or (row["result"] == -1 and s2 < s1))
        return strict, float(strict)

    series_df[["upset_strict", "upset_weak"]] = pd.DataFrame(
        series_df.apply(get_upsets, axis=1).tolist(), index=series_df.index
    )
    upset_counts = series_df.groupby(keys, as_index=False)[["upset_strict", "upset_weak"]].sum()

    score_sequences = score_sequences.sort_values(
        keys + ["wins", "team"], ascending=[True, True, True, False, True]
    )
    scoring_summary = score_sequences.groupby(keys, as_index=False).agg(
        num_teams=("team", "size"), teams=("team", tuple),
        scoring_sequence=("wins", tuple)
    )
    games_per_pair = series_df.groupby(keys, as_index=False)["games_played"].first()
    games_per_pair = games_per_pair.rename(columns={"games_played": "games_per_pair"})

    analysis_df = scoring_summary.merge(games_per_pair, on=keys).merge(upset_counts, on=keys)
    analysis_df["num_games"] = analysis_df["num_teams"].apply(lambda n: comb(int(n), 2))
    analysis_df["num_admissible"] = analysis_df["num_games"] - analysis_df["scoring_sequence"].apply(
        lambda scores: sum(comb(count, 2) for count in Counter(scores).values())
    )
    denominator = analysis_df["num_admissible"].astype(float)
    denominator = denominator.where(denominator > 0)
    analysis_df["upset_strict_frequency"] = analysis_df["upset_strict"] / denominator
    analysis_df["upset_weak_frequency"] = analysis_df["upset_weak"] / denominator

    # Preserve the existing CSV schema, with four additional columns.
    # Keep league only if needed to distinguish otherwise identical row keys.
    columns = ["season", "division", "num_teams", "games_per_pair", "teams",
               "scoring_sequence", "upset_strict", "upset_weak", "num_games",
               "num_admissible", "upset_strict_frequency", "upset_weak_frequency"]
    if analysis_df.duplicated(["season", "division"]).any():
        columns.insert(1, "league")
    analysis_df = analysis_df[columns]
    output_path = Path(output_file)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    analysis_df.to_csv(output_path, index=False)
    print(f"Saved {len(analysis_df)} rows to {output_path}")
    return analysis_df


In [3]:
# Bundesliga: edit these paths for another league.
analysis_df = generate_analysis(
    series_file="../Data/bund_division_series_results.csv",
    membership_file="../Data/bund_division_membership.csv",
    output_file="../Analysis/bundesliga_analysis.csv",
    tie_score=0.5,
)

# Premier League:
# analysis_df = generate_analysis(
#     series_file="../Data/prem_division_series_results.csv",
#     membership_file="../Data/prem_division_membership.csv",
#     output_file="../Analysis/premier_league_analysis.csv",
#     tie_score=0.5,
# )

# Existing MLB / NBA / NFL (replace nba with mlb or nfl):
# analysis_df = generate_analysis(
#     series_file="../Data/nba_division_series_results.csv",
#     output_file="../Analysis/nba_analysis.csv",
#     tie_score=0,
# )


Saved 24 rows to ../Analysis/bundesliga_analysis.csv


In [4]:
# printing if u want
analysis_df

,season,division,num_teams,games_per_pair,teams,scoring_sequence,upset_strict,upset_weak,num_games,num_admissible,upset_strict_frequency,upset_weak_frequency
0,2001,Bundesliga,18,2,"(BAY, DOR, LEV, HER, SCH, BRE, KAI, MUN, STU, ...","(13.0, 12.5, 12.5, 12.0, 12.0, 9.5, 9.5, 9.5, ...",16,38.5,153,142,0.112676,0.271127
1,2002,Bundesliga,18,2,"(BAY, DOR, HSV, STU, HER, SCH, BRE, WOL, BOC, ...","(13.5, 11.5, 11.0, 11.0, 9.5, 9.0, 8.5, 8.5, 8...",26,54.5,153,143,0.181818,0.381119
2,2003,Bundesliga,18,2,"(BRE, BAY, LEV, STU, BOC, DOR, SCH, HSV, HER, ...","(14.5, 13.5, 12.5, 11.5, 11.0, 9.5, 9.5, 9.0, ...",13,39.5,153,147,0.088435,0.268707
3,2004,Bundesliga,18,2,"(BAY, HER, DOR, LEV, SCH, STU, BRE, HSV, WOL, ...","(14.0, 12.0, 11.0, 11.0, 11.0, 10.5, 10.0, 9.0...",17,43.5,153,145,0.117241,0.300000
4,2005,Bundesliga,18,2,"(BAY, BRE, SCH, HSV, DOR, LEV, HER, NUR, HAN, ...","(15.5, 13.0, 12.5, 11.5, 9.5, 9.5, 9.0, 8.0, 7...",22,45.0,153,147,0.149660,0.306122
5,2006,Bundesliga,18,2,"(STU, SCH, BRE, BAY, HSV, NUR, BIE, HER, HAN, ...","(13.0, 12.5, 12.0, 10.5, 10.0, 9.5, 9.0, 8.5, ...",25,50.5,153,147,0.170068,0.343537
6,2007,Bundesliga,18,2,"(BAY, BRE, SCH, HSV, WOL, HAN, LEV, STU, BOC, ...","(15.5, 11.5, 11.5, 11.0, 10.0, 9.5, 9.5, 8.5, ...",27,46.0,153,146,0.184932,0.315068
7,2008,Bundesliga,18,2,"(WOL, BAY, DOR, STU, HOF, HSV, HER, SCH, LEV, ...","(13.5, 12.5, 12.0, 11.5, 11.0, 11.0, 10.5, 9.0...",22,46.5,153,149,0.147651,0.312081
8,2009,Bundesliga,18,2,"(BAY, LEV, SCH, BRE, STU, DOR, HSV, MAI, FRA, ...","(13.5, 12.0, 12.0, 11.5, 11.5, 11.0, 10.0, 9.0...",23,44.5,153,148,0.155405,0.300676
9,2010,Bundesliga,18,2,"(DOR, BAY, LEV, HAN, MAI, FRE, HOF, KAI, NUR, ...","(14.0, 13.0, 12.5, 10.5, 10.0, 8.0, 8.0, 8.0, ...",19,41.0,153,140,0.135714,0.292857
